<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML321ENSkillsNetwork817-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Classification-based Rating Mode Prediction using Embedding Features**


Estimated time needed: **60** minutes


In this lab, you have built regression models to predict numerical course ratings using the embedding feature vectors extracted from neural networks. We can also consider the prediction problem as a classification problem also using embedding features.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/module_4/images/rating_classification.png)


The workflow is very similar to our previous lab. We first extract two embedding matrices out of the neural network, and aggregate them to be a single interaction feature vector as input data `X`.

This time, with the interaction label `Y` as categorical rating mode, we can build classification models to approximate the mapping from `X` to `Y`, as shown in the above flowchart.


## Objectives


After completing this lab you will be able to:


* Build classification models to predict rating modes using the combined embedding vectors


----


## Prepare and setup lab environment


First install and import required libraries:


In [1]:
%pip install scikit-learn
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# also set a random state
rs = 123

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

### Load datasets


In [4]:
rating_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-ML0321EN-Coursera/labs/v2/module_3/ratings.csv"
# Embeddings produced by lab_jupyter_cf_ann.ipynb (produce-your-own route)
user_emb_url = "../data/user_embeddings.csv"
item_emb_url = "../data/course_embeddings.csv"

The first dataset is the rating dataset contains user-item interaction matrix


In [5]:
rating_df = pd.read_csv(rating_url)

In [6]:
rating_df.head()

,user,item,rating
0,1889878,CC0101EN,5
1,1342067,CL0101EN,3
2,1990814,ML0120ENv3,5
3,380098,BD0211EN,5
4,779563,DS0101EN,3


As you can see from the above data, the user and item are just ids, let's substitute them with their embedding vectors


In [7]:
user_emb = pd.read_csv(user_emb_url)
item_emb = pd.read_csv(item_emb_url)

In [8]:
user_emb.head()

,user,UFeature0,UFeature1,UFeature2,UFeature3,UFeature4,UFeature5,UFeature6,UFeature7,UFeature8,UFeature9,UFeature10,UFeature11,UFeature12,UFeature13,UFeature14,UFeature15
0,1889878,-0.169329,-0.171417,0.077784,-0.095310,0.182348,-0.077800,-0.087430,0.056138,-0.131548,0.036847,-0.063690,0.013257,-0.125162,0.068581,0.057402,-0.175451
1,1342067,-0.101114,0.000926,0.023026,0.002143,0.077819,-0.076868,-0.087827,-0.071640,-0.071966,0.015049,0.007795,-0.019745,-0.089747,0.115308,0.035302,0.091108
2,1990814,0.048882,0.242381,0.142094,-0.023209,0.214116,0.083913,-0.126982,-0.125498,0.016538,-0.199776,0.065957,-0.011468,-0.025896,-0.022052,-0.062476,-0.001740
3,380098,0.125124,0.091391,-0.064431,0.006099,0.121973,-0.050488,-0.122787,0.053204,-0.057795,0.040821,-0.026236,-0.002356,-0.109218,-0.059387,0.149408,0.113286
4,779563,0.061062,-0.127827,-0.072394,-0.026350,-0.066526,-0.011174,0.042442,-0.096202,0.134940,0.101634,0.006678,-0.024174,0.067476,-0.179815,0.072761,0.133266


In [9]:
item_emb.head()

,item,CFeature0,CFeature1,CFeature2,CFeature3,CFeature4,CFeature5,CFeature6,CFeature7,CFeature8,CFeature9,CFeature10,CFeature11,CFeature12,CFeature13,CFeature14,CFeature15
0,CC0101EN,-0.008491,0.009244,0.001581,0.026333,0.008334,0.019389,0.003154,-0.035585,-0.006947,-0.016610,0.033818,0.007301,0.016022,-0.013650,0.000223,-0.001176
1,CL0101EN,-0.029734,0.004281,0.030478,-0.043487,-0.029202,-0.030779,0.046974,0.015391,0.038814,0.000829,-0.004464,-0.022572,0.055641,-0.029218,0.015049,0.011639
2,ML0120ENv3,-0.016919,-0.010686,-0.010309,-0.042168,0.001871,-0.002098,-0.039025,-0.043623,-0.034600,-0.055885,-0.019527,-0.029323,-0.010548,-0.023231,0.038712,0.001239
3,BD0211EN,0.017437,-0.008801,0.032791,0.000298,-0.011701,-0.000531,-0.007562,0.040755,-0.005233,0.003373,-0.027239,-0.009436,0.006211,0.001653,0.013483,0.026129
4,DS0101EN,0.003147,-0.030302,-0.015159,-0.002713,0.004832,0.027061,-0.001908,-0.004655,-0.008233,0.005730,0.005575,0.016058,-0.007969,0.005005,-0.002917,-0.002826


In [10]:
# Merge user embedding features
merged_df = pd.merge(rating_df, user_emb, how='left', left_on='user', right_on='user').fillna(0)
# Merge course embedding features
merged_df = pd.merge(merged_df, item_emb, how='left', left_on='item', right_on='item').fillna(0)

In [11]:
merged_df.head()

,user,item,rating,UFeature0,UFeature1,UFeature2,UFeature3,UFeature4,UFeature5,UFeature6,...,CFeature6,CFeature7,CFeature8,CFeature9,CFeature10,CFeature11,CFeature12,CFeature13,CFeature14,CFeature15
0,1889878,CC0101EN,5,-0.169329,-0.171417,0.077784,-0.095310,0.182348,-0.077800,-0.087430,...,0.003154,-0.035585,-0.006947,-0.016610,0.033818,0.007301,0.016022,-0.013650,0.000223,-0.001176
1,1342067,CL0101EN,3,-0.101114,0.000926,0.023026,0.002143,0.077819,-0.076868,-0.087827,...,0.046974,0.015391,0.038814,0.000829,-0.004464,-0.022572,0.055641,-0.029218,0.015049,0.011639
2,1990814,ML0120ENv3,5,0.048882,0.242381,0.142094,-0.023209,0.214116,0.083913,-0.126982,...,-0.039025,-0.043623,-0.034600,-0.055885,-0.019527,-0.029323,-0.010548,-0.023231,0.038712,0.001239
3,380098,BD0211EN,5,0.125124,0.091391,-0.064431,0.006099,0.121973,-0.050488,-0.122787,...,-0.007562,0.040755,-0.005233,0.003373,-0.027239,-0.009436,0.006211,0.001653,0.013483,0.026129
4,779563,DS0101EN,3,0.061062,-0.127827,-0.072394,-0.026350,-0.066526,-0.011174,0.042442,...,-0.001908,-0.004655,-0.008233,0.005730,0.005575,0.016058,-0.007969,0.005005,-0.002917,-0.002826


Each user's embedding features and each item's embedding features are added to the dataset. Next, we perform element-wise add the user features (the column labels starting with `UFeature`) and item features (the column labels starting with `CFeature`).


In [12]:
u_feautres = [f"UFeature{i}" for i in range(16)] # Assuming there are 16 user embedding features
c_features = [f"CFeature{i}" for i in range(16)] # Assuming there are 16 course embedding features
# Extract user embedding features
user_embeddings = merged_df[u_feautres]
# Extract course embedding features
course_embeddings = merged_df[c_features]
# Extract ratings
ratings = merged_df['rating']

# Aggregate the two feature columns using element-wise add
interaction_dataset = user_embeddings + course_embeddings.values
# Rename the columns of the resulting DataFrame
interaction_dataset.columns = [f"Feature{i}" for i in range(16)]
# Add the 'rating' column from the original DataFrame to the regression dataset
interaction_dataset['rating'] = ratings
# Display the first few rows of the regression dataset
interaction_dataset.head()

,Feature0,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10,Feature11,Feature12,Feature13,Feature14,Feature15,rating
0,-0.177820,-0.162174,0.079365,-0.068978,0.190682,-0.058411,-0.084276,0.020553,-0.138495,0.020237,-0.029873,0.020558,-0.109140,0.054931,0.057625,-0.176627,5
1,-0.130848,0.005207,0.053504,-0.041344,0.048617,-0.107647,-0.040853,-0.056250,-0.033151,0.015878,0.003331,-0.042317,-0.034107,0.086089,0.050351,0.102747,3
2,0.031963,0.231696,0.131785,-0.065377,0.215987,0.081816,-0.166008,-0.169121,-0.018062,-0.255661,0.046429,-0.040791,-0.036444,-0.045283,-0.023765,-0.000501,5
3,0.142561,0.082590,-0.031639,0.006397,0.110272,-0.051019,-0.130349,0.093959,-0.063027,0.044194,-0.053474,-0.011792,-0.103006,-0.057734,0.162891,0.139415,5
4,0.064209,-0.158129,-0.087553,-0.029063,-0.061694,0.015887,0.040534,-0.100857,0.126708,0.107364,0.012253,-0.008116,0.059507,-0.174809,0.069844,0.130440,3


Next, let's use `LabelEncoder()` to encode our `rating` label to be categorical:


In [13]:
# Extract features (X) from the interaction_dataset DataFrame
# Selects all rows and all columns except the last column (features)
X = interaction_dataset.iloc[:, :-1]
# Extract the target variable (y_raw) from the interaction_dataset DataFrame
# Selects all rows and only the last column (target variable)
y_raw = interaction_dataset.iloc[:, -1]
# Initialize a LabelEncoder object to encode the target variable
label_encoder = LabelEncoder()
# Encode the target variable (y_raw) using the LabelEncoder
# .values.ravel() converts the target variable to a flattened array before encoding
# The LabelEncoder fits and transforms the target variable, assigning encoded labels to y
y = label_encoder.fit_transform(y_raw.values.ravel())

**Note on "rating mode" here:** this lab's original framing predicts a binary audit-vs-complete mode, which assumed a `{2, 3}` rating scale. This pipeline is standardized on `module_3/ratings.csv`, whose ratings are actually a 3-point quality scale, `{3, 4, 5}` — there is no audit/complete distinction in this data. So `y` is a **3-class target** (rating 3, 4, or 5), not binary, and there's no single "positive" class to score against for ranking.

Instead, for the predict-then-rank step below, `SCORE` is the **expected rating**: `predict_proba()`'s per-class probabilities dotted with their actual rating values (3, 4, 5), giving a single continuous score per (user, course) pair. This also makes this notebook's output directly comparable to `cf_regression_w_embeddings.ipynb`, which predicts the same expected quantity via regression instead of classification.

and split X and y into training and testing dataset:


In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=rs)

In [15]:
print(f"Input data shape: {X.shape}, Output data shape: {y.shape}")

Input data shape: (233306, 16), Output data shape: (233306,)


## TASK: Perform classification tasks on the interaction dataset


Now our input data `X` and output label `y` is ready, let's build classification models to map `X` to `y`


You may use `sklearn` to train and evaluate various regression models.


_TODO: Define classification models such as Logistic Regression, Tree models, SVM, Bagging, and Boosting models_


In [16]:
model = RandomForestClassifier(max_depth=10, random_state=rs, n_jobs=-1)

<details>
    <summary>Click here for Hints </summary>
    
For Example: you can call `RandomForestClassifier()` to define your model, don't forget to specify `max_depth= ..`  and `random_state=rs` in the parameters.


_TODO: Train your classification models with training data_


In [17]:
model.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",123
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total 

<details>
    <summary>Click here for Hints</summary>
    
You can call `model.fit()` method with `X_train, y_train` parameters.


_TODO: Evaluate your classification models_


In [18]:
# Class distribution (rating values, not encoded labels) -- check how balanced the 3 classes are
class_counts = y_raw.value_counts().sort_index()
class_props = (class_counts / class_counts.sum() * 100).round(2)
print("Class distribution (rating -> count, % of total):")
for rating_value, count in class_counts.items():
    print(f"  {rating_value}: {count} ({class_props[rating_value]}%)")

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_test, y_pred, average="macro")
precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted")

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Macro    - precision: {precision_macro:.4f}, recall: {recall_macro:.4f}, F1: {f1_macro:.4f}")
print(f"Weighted - precision: {precision_weighted:.4f}, recall: {recall_weighted:.4f}, F1: {f1_weighted:.4f}")

Class distribution (rating -> count, % of total):
  3: 77866 (33.38%)
  4: 77936 (33.41%)
  5: 77504 (33.22%)



Accuracy: 0.3362
Macro    - precision: 0.3380, recall: 0.3376, F1: 0.3290
Weighted - precision: 0.3381, recall: 0.3362, F1: 0.3283


In [19]:
import os
import numpy as np

# Predict-then-rank. Score every (user, unseen-course) pair via a single vectorized
# predict_proba() call -- not a per-pair loop, which would be ~33,901 x 126 = 4.3M calls.
# SCORE = expected rating = predict_proba() columns dotted with their real rating values,
# read off label_encoder.classes_[model.classes_] rather than hardcoded, since predict_proba's
# column order follows model.classes_ (the label-encoded values), not the raw ratings.
user_ids = user_emb["user"].tolist()
course_ids = item_emb["item"].tolist()
user_matrix = user_emb[u_feautres].to_numpy(dtype="float32")
course_matrix = item_emb[c_features].to_numpy(dtype="float32")

num_users = len(user_ids)
num_courses = len(course_ids)

# Element-wise-add every user embedding with every course embedding (same aggregation as
# interaction_dataset above), flattened to a single (num_users * num_courses, 16) batch
interaction_matrix = user_matrix[:, None, :] + course_matrix[None, :, :]
interaction_matrix = interaction_matrix.reshape(-1, interaction_matrix.shape[-1])

probas = model.predict_proba(interaction_matrix)
rating_values_per_column = label_encoder.classes_[model.classes_]
expected_rating = probas @ rating_values_per_column
score_matrix = expected_rating.reshape(num_users, num_courses)

# Mask out courses each user has already rated, restricted to this model's 126-course universe
enrolled_pivot = rating_df.assign(_v=1).pivot_table(index="user", columns="item", values="_v", aggfunc="max")
enrolled_pivot = enrolled_pivot.reindex(index=user_ids, columns=course_ids, fill_value=0).fillna(0)
enrolled_mask = enrolled_pivot.to_numpy() > 0

score_matrix_masked = np.where(enrolled_mask, -np.inf, score_matrix)

TOP_N_PER_USER = 20
top_n_idx = np.argsort(-score_matrix_masked, axis=1)[:, :TOP_N_PER_USER]

rows = []
for u_idx, user_id in enumerate(user_ids):
    for c_idx in top_n_idx[u_idx]:
        score = score_matrix_masked[u_idx, c_idx]
        if np.isfinite(score):
            rows.append((user_id, course_ids[c_idx], score))

cf_classification_res_df = pd.DataFrame(rows, columns=["USER", "COURSE_ID", "SCORE"])

os.makedirs("../data/predictions", exist_ok=True)
cf_classification_res_df.to_csv("../data/predictions/cf_classification.csv", index=False)
print(f"Saved {cf_classification_res_df.shape[0]} rows to data/predictions/cf_classification.csv "
      f"({cf_classification_res_df['USER'].nunique()} users)")

D:\ibm ml capstone\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Saved 678020 rows to data/predictions/cf_classification.csv (33901 users)


<details>
    <summary>Click here for Hints</summary>
    
You can call `model.predict()` method with `X_test` parameter to get model predictions. Then use `accuracy_score()` with `y_test, your_predictions` parameters to calculate the accuracy value. 
* You can use `precision_recall_fscore_support` command  with `y_test, your_predictions, average='binary'` parameters get recall, precision and F score.
    


### Summary


In this lab, you have built and evaluated various classification models to predict categorical course rating modes using the embedding feature vectors extracted from neural networks.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


```toggle## Change Log
```


```toggle|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
```
```toggle|-|-|-|-|
```
```toggle|2021-10-25|1.0|Yan|Created the initial version|
```


Copyright © 2021 IBM Corporation. All rights reserved.
